# LifeMate — Qwen3.5-4B SFT continuation

Tiếp tục từ **adapter epoch 3, English system prompt**, fine-tune trên 7.746 dòng (3.873 gốc + 3.873 synthetic). ZIP trên Drive chứa dữ liệu và adapter; model nền tải trực tiếp từ Hugging Face.

Mặc định: QLoRA NF4, LoRA r=16/alpha=32, 1 epoch, LR 1e-5, max length 8.192, effective batch 16. Optimizer và scheduler mới được khởi tạo; notebook tiếp tục từ adapter weights epoch 3, không khôi phục optimizer state.

Điểm public 47.320 là kết quả end-to-end người dùng báo cáo cho pipeline PhoWhisper + epoch 3 English + CoT + focused post-lookup prompt. Đây không phải điểm riêng của adapter hay lần SFT synthetic mới. CoT và focused prompt chỉ dùng inference; target SFT là action JSON, không có CoT.

Chọn Colab GPU L4, bật Internet, upload ZIP vào MyDrive/lifemate, chỉnh CFG bên dưới nếu cần, rồi chạy từ trên xuống. Đổi RUN_NAME trước mỗi lần chạy mới.

In [ ]:
from pathlib import Path
# Sửa đường dẫn và tham số tại đây.
CFG = {
 "DRIVE_MOUNT":"/content/drive", "DRIVE_ROOT":"/content/drive/MyDrive/lifemate",
 "BUNDLE_NAME":"lifemate-qwen35-pho-synthetic-epoch3-continue-colab-v1.zip",
 "EXTRACT_ROOT":"/content/lifemate-qwen35-epoch3-sft",
 "RUN_NAME":"qwen35-pho-synthetic-epoch3-continue-v1", "HF_CACHE_DIR":"/content/hf_cache",
 "MODEL_ID":"Qwen/Qwen3.5-4B", "MODEL_REVISION":"main",
 "DATA_REL":"data/combined_sft_7746.jsonl", "MANIFEST_REL":"data/dataset_manifest.json",
 "ADAPTER_REL":"initial_adapter",
 "DATA_SHA256":"6203ce35636f65a34b629b0398f12cc2bf1b8bdbd3c266a1abf5a5602ffc16d1",
 "ADAPTER_SHA256":"a15c99d66b1bc2c14846428ef0efd84865110d084fbf17f7206517ad69977048",
 "ROWS":7746, "TRAJECTORIES":1236,
 "EPOCHS":1.0, "LR":1e-5, "MAX_LENGTH":8192, "EFFECTIVE_BATCH":16,
 "PER_DEVICE_BATCH":1, "WARMUP_RATIO":0.05, "SAVE_STEPS":60, "SAVE_LIMIT":5,
 "SEED":20261106, "LORA_R":16, "LORA_ALPHA":32, "LORA_DROPOUT":0.05,
 "TRANSFORMERS":"5.22.1", "PEFT":"0.21.2", "ACCELERATE":"1.15.0", "BNB":"0.50.2",
 "REQUIRE_L4":True, "MAX_TRAIN_ROWS":None,
}
CFG["BUNDLE_PATH"]=str(Path(CFG["DRIVE_ROOT"])/CFG["BUNDLE_NAME"])
CFG["OUTPUT_DIR"]=str(Path(CFG["DRIVE_ROOT"])/"colab-runs"/CFG["RUN_NAME"])
print(CFG)

In [ ]:
from google.colab import drive
drive.mount(CFG["DRIVE_MOUNT"])
BUNDLE=Path(CFG["BUNDLE_PATH"])
if not BUNDLE.is_file(): raise FileNotFoundError(f"Upload ZIP to Drive: {BUNDLE}")
%pip install -q transformers==5.22.1 peft==0.21.2 accelerate==1.15.0 bitsandbytes==0.50.2

In [ ]:
import hashlib, importlib.metadata as md, json, os, random, zipfile
import torch

for p,v in {"transformers":CFG["TRANSFORMERS"],"peft":CFG["PEFT"],
            "accelerate":CFG["ACCELERATE"],"bitsandbytes":CFG["BNB"]}.items():
    if md.version(p)!=v: raise RuntimeError(f"{p}: expected {v}, got {md.version(p)}")
if not torch.cuda.is_available(): raise RuntimeError("Enable Colab GPU.")
GPU=torch.cuda.get_device_name(0)
if CFG["REQUIRE_L4"] and "L4" not in GPU.upper(): raise RuntimeError(f"Expected L4, got {GPU}")
print({"gpu":GPU,"vram_gib":round(torch.cuda.get_device_properties(0).total_memory/2**30,1),
       "bf16":torch.cuda.is_bf16_supported()})

def sha_file(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for b in iter(lambda:f.read(8*1024*1024),b""): h.update(b)
    return h.hexdigest()

ROOT=Path(CFG["EXTRACT_ROOT"]).resolve(); ROOT.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z:
    if z.testzip(): raise RuntimeError("Corrupt ZIP")
    for x in z.infolist():
        q=(ROOT/x.filename).resolve()
        if ROOT not in q.parents and q!=ROOT: raise RuntimeError("Unsafe ZIP path")
    z.extractall(ROOT)
PKG=ROOT/"lifemate-qwen35-pho-synthetic-epoch3-continue-colab-v1"
DATA=PKG/CFG["DATA_REL"]; MANIFEST=PKG/CFG["MANIFEST_REL"]; ADAPTER=PKG/CFG["ADAPTER_REL"]
if sha_file(DATA)!=CFG["DATA_SHA256"] or sha_file(ADAPTER/"adapter_model.safetensors")!=CFG["ADAPTER_SHA256"]:
    raise RuntimeError("Dataset/adapter checksum mismatch")
M=json.loads(MANIFEST.read_text(encoding="utf-8"))
with DATA.open(encoding="utf-8") as f: ROWS=[json.loads(x) for x in f if x.strip()]
assert len(ROWS)==CFG["ROWS"]==M["combined_sft_rows"]
assert len({r["sample_id"] for r in ROWS})==CFG["TRAJECTORIES"]
assert M["validation_ids_excluded_from_training"]==309 and M["train_validation_overlap"]==0
assert M["toolenv_gold_replay"]["failures"]==0
assert all(len(r["messages"])==2 and r["messages"][0]["role"]=="system"
           and r["messages"][1]["role"]=="user" and r["target"]["action"] in {"tool_call","stop"} for r in ROWS)
PROMPTS={r["messages"][0]["content"] for r in ROWS}
assert len(PROMPTS)==1 and "You are LifeMate, a Vietnamese-language agent" in next(iter(PROMPTS))
PROMPT_SHA=hashlib.sha256(next(iter(PROMPTS)).encode()).hexdigest()
if CFG["MAX_TRAIN_ROWS"] is not None: ROWS=ROWS[:int(CFG["MAX_TRAIN_ROWS"])]
AC=json.loads((ADAPTER/"adapter_config.json").read_text())
assert AC["base_model_name_or_path"]==CFG["MODEL_ID"] and AC["r"]==CFG["LORA_R"]
print({"rows":len(ROWS),"train_ids":len({r["sample_id"] for r in ROWS}),
       "data_sha":CFG["DATA_SHA256"],"adapter_sha":CFG["ADAPTER_SHA256"],
       "sft_system_prompt_sha":PROMPT_SHA,"ToolEnv_failures":0})

## Tải model nền trực tiếp

Model nền không nằm trong ZIP. Cell sau tải Qwen3.5-4B theo model ID/revision đã khai báo; cache lưu trên ổ tạm Colab.

In [ ]:
from transformers import AutoTokenizer, AutoModelForMultimodalLM, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training
import transformers

os.environ["HF_HOME"]=CFG["HF_CACHE_DIR"]
os.environ["TOKENIZERS_PARALLELISM"]="false"; os.environ["WANDB_DISABLED"]="true"
random.seed(CFG["SEED"]); torch.manual_seed(CFG["SEED"]); torch.cuda.manual_seed_all(CFG["SEED"])
TOK=AutoTokenizer.from_pretrained(CFG["MODEL_ID"],revision=CFG["MODEL_REVISION"],
                                  cache_dir=CFG["HF_CACHE_DIR"],use_fast=True)
if TOK.pad_token_id is None: TOK.pad_token=TOK.eos_token
TOK.padding_side="right"
if TOK.eos_token_id is None: raise RuntimeError("Tokenizer has no EOS token.")
DTYPE=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=DTYPE)
kw={"quantization_config":bnb,"device_map":{"":0},"low_cpu_mem_usage":True,
    "attn_implementation":"sdpa","cache_dir":CFG["HF_CACHE_DIR"],"revision":CFG["MODEL_REVISION"]}
kw["dtype" if int(transformers.__version__.split(".")[0])>=5 else "torch_dtype"]=DTYPE
print("Downloading",CFG["MODEL_ID"],CFG["MODEL_REVISION"])
base=AutoModelForMultimodalLM.from_pretrained(CFG["MODEL_ID"],**kw)
base.config.use_cache=False
base=prepare_model_for_kbit_training(base,use_gradient_checkpointing=True,
                                      gradient_checkpointing_kwargs={"use_reentrant":False})
model=PeftModel.from_pretrained(base,str(ADAPTER),is_trainable=True)
TRAINABLE=sum(p.numel() for p in model.parameters() if p.requires_grad)
if TRAINABLE==0: raise RuntimeError("No trainable LoRA parameters.")
model.print_trainable_parameters()

In [ ]:
# Thinking template được giữ; loss chỉ tính trên action JSON gold, không train CoT.
CLOSE=TOK.encode("</think>\n\n",add_special_tokens=False)
ENCODED=[]; LENGTHS=[]; TRUNCATED=0
for r in ROWS:
    p=TOK.apply_chat_template(r["messages"],tokenize=False,add_generation_prompt=True,enable_thinking=True)
    pids=TOK.encode(p,add_special_tokens=False)
    ans=TOK.encode(json.dumps(r["target"],ensure_ascii=False,separators=(",",":")),add_special_tokens=False)+[TOK.eos_token_id]
    room=CFG["MAX_LENGTH"]-len(CLOSE)-len(ans)
    if room<1: raise ValueError(f"Target too long: {r['sample_id']}/{r['step_index']}")
    if len(pids)>room:
        TRUNCATED+=1; head=int(room*.72); pids=pids[:head]+pids[-(room-head):]
    ids=pids+CLOSE+ans
    ENCODED.append({"input_ids":ids,"labels":[-100]*(len(pids)+len(CLOSE))+ans})
    LENGTHS.append(len(ids))
TOK_PATH=Path("/content/lifemate_qwen35_tokens.pt"); torch.save(ENCODED,TOK_PATH)
print({"rows":len(ENCODED),"max_len":max(LENGTHS),"truncated":TRUNCATED,
       "token_cache_mib":round(TOK_PATH.stat().st_size/2**20,1)})

## Huấn luyện và lưu kết quả

Checkpoint lưu vào Drive theo chu kỳ. Optimizer/scheduler mới. Nếu runtime bị ngắt, checkpoint nằm trong output; notebook không tự khôi phục optimizer state.

In [ ]:
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments

class SFTDataset(Dataset):
    def __len__(self): return len(ENCODED)
    def __getitem__(self,i): return ENCODED[i]

class Collator:
    def __call__(self,features):
        width=((max(len(x["input_ids"]) for x in features)+7)//8)*8
        ids=[]; masks=[]; labels=[]
        for x in features:
            pad=width-len(x["input_ids"])
            ids.append(x["input_ids"]+[TOK.pad_token_id]*pad)
            masks.append([1]*len(x["input_ids"])+[0]*pad)
            labels.append(x["labels"]+[-100]*pad)
        lab=torch.tensor(labels,dtype=torch.long)
        if lab.shape[0]!=1: raise RuntimeError("Use per-device batch size 1.")
        shifted=lab[:,1:]; mask=shifted.ne(-100)
        keep=torch.where(mask[0])[0]; shift_labels=shifted[mask]
        if keep.numel()==0: raise RuntimeError("No target tokens.")
        return {"input_ids":torch.tensor(ids,dtype=torch.long),
                "attention_mask":torch.tensor(masks,dtype=torch.long),
                "labels":lab,"logits_to_keep":keep,"shift_labels":shift_labels}

collate=Collator()
idx=max(range(len(ENCODED)),key=lambda i:len(ENCODED[i]["input_ids"]))
probe={k:v.to(0) for k,v in collate([ENCODED[idx]]).items()}
loss=model(**probe).loss
if not torch.isfinite(loss): raise RuntimeError("QLoRA preflight failed.")
loss.backward(); model.zero_grad(set_to_none=True); del probe,loss; torch.cuda.empty_cache()

OUT=Path(CFG["OUTPUT_DIR"])
if OUT.exists() and any(OUT.iterdir()): raise RuntimeError(f"Output exists; change RUN_NAME: {OUT}")
OUT.mkdir(parents=True,exist_ok=True); CKPT=OUT/"checkpoints"
accum=max(1,(CFG["EFFECTIVE_BATCH"]+CFG["PER_DEVICE_BATCH"]-1)//CFG["PER_DEVICE_BATCH"])
steps=(len(ENCODED)+CFG["EFFECTIVE_BATCH"]-1)//CFG["EFFECTIVE_BATCH"]
warm=max(1,round(.05*steps))
args=TrainingArguments(output_dir=str(CKPT),num_train_epochs=CFG["EPOCHS"],
    per_device_train_batch_size=CFG["PER_DEVICE_BATCH"],gradient_accumulation_steps=accum,
    learning_rate=CFG["LR"],weight_decay=.01,warmup_steps=warm,lr_scheduler_type="cosine",
    logging_steps=10,save_strategy="steps",save_steps=CFG["SAVE_STEPS"],
    save_total_limit=CFG["SAVE_LIMIT"],optim="adamw_torch",
    bf16=DTYPE==torch.bfloat16,fp16=DTYPE==torch.float16,
    gradient_checkpointing=True,gradient_checkpointing_kwargs={"use_reentrant":False},
    remove_unused_columns=False,report_to=[],seed=CFG["SEED"],data_seed=CFG["SEED"],dataloader_num_workers=0)
trainer=Trainer(model=model,args=args,train_dataset=SFTDataset(),data_collator=collate)
print({"rows":len(ENCODED),"steps_per_epoch":steps,"warmup_steps":warm,
       "effective_batch":accum*CFG["PER_DEVICE_BATCH"],"checkpoint_dir":str(CKPT)})
result=trainer.train()

FINAL=OUT/"adapter_final"; trainer.save_model(str(FINAL)); TOK.save_pretrained(FINAL)
W=FINAL/"adapter_model.safetensors"
if not W.is_file():
    candidates=list(FINAL.glob("adapter_model*.safetensors"))
    if len(candidates)!=1: raise RuntimeError("Final adapter weights missing.")
    W=candidates[0]
report={"run_type":"continue_epoch3_english_adapter_phowhisper_plus_synthetic",
 "initial_adapter_sha256":CFG["ADAPTER_SHA256"],"initial_epoch":3,
 "sft_data_sha256":CFG["DATA_SHA256"],"sft_prompt_sha256":PROMPT_SHA,
 "sft_prompt_variant":"english-system-prompt-v1","cot_targets_added":False,
 "train_rows":len(ENCODED),"epochs":CFG["EPOCHS"],"learning_rate":CFG["LR"],
 "optimizer_state":"fresh","scheduler_state":"fresh cosine schedule","warmup_steps":warm,
 "effective_batch_size":accum*CFG["PER_DEVICE_BATCH"],"max_seq_length":CFG["MAX_LENGTH"],
 "base_model":CFG["MODEL_ID"],"base_revision":CFG["MODEL_REVISION"],"gpu":GPU,
 "train_runtime_seconds":result.metrics.get("train_runtime"),"train_loss":result.metrics.get("train_loss"),
 "final_adapter_sha256":sha_file(W),
 "public_score_note":"47.320 is a prior end-to-end inference score, not this SFT result."}
(OUT/"training_report.json").write_text(json.dumps(report,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
(OUT/"run_config.json").write_text(json.dumps(CFG,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
print(json.dumps(report,ensure_ascii=False,indent=2)); print("Adapter saved:",FINAL)

Provenance note: báo cáo train epoch 3 không lưu hash chính xác của snapshot model nền Kaggle. Model ID và revision Hugging Face có thể chỉnh ở CFG; hãy xác minh revision nếu cần tái lập tuyệt đối.